# Event Grammar

Purpose: understand how HALO encodes shots, deflections, blocks, assists, and goals before building an analytical shot-value dataset.

This notebook follows `01_data_inventory.ipynb`. It goes deeper than a generic inventory because shot value depends on correctly interpreting multi-row hockey events.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Resolve project paths whether the notebook is launched from the project root
# or from inside the notebooks folder.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
HALO_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"

print("Project root:", PROJECT_ROOT)
print("HALO raw folder exists:", HALO_RAW.exists())

Project root: c:\Users\rinal\OneDrive\Documents\hockey-analytics\outside-shot-value
HALO raw folder exists: True


In [3]:
# Load only the event table for this notebook.
# The tracking and stint tables matter later, but event grammar can be inspected
# from the event log alone.

events = pd.read_parquet(HALO_RAW / "events.parquet")

print("events shape:", events.shape)

events shape: (1800464, 24)


## Shot and Goal Representation

Initial inspection suggests that goals are represented across multiple rows:

1. A `shot` row with `outcome == "successful"` and a non-null `sl_xg_all_shots`.
2. A player-level `goal` row shortly after the shot.
3. A game-level `goal` row with missing player/team fields.

For modeling shot value, the shot row should likely be treated as the scoring chance. Goal rows should be used to label outcomes, not counted as additional shot attempts.

## Deflections, Blocks, and Blocked Shots

The HALO documentation distinguishes between:

- `shot` rows with details such as `outsideblocked` and `slotblocked`
- `deflection` rows
- `block` rows, described as pass or shot block attempts

This creates an important modeling question: are deflections treated as secondary offensive shot events, while blocked shots are failed shot attempts, or do these categories overlap?

We will not assume the answer. We will compare timing, sequence IDs, coordinates, outcomes, and xG patterns across `shot`, `deflection`, and `block` rows.

In [4]:
# Compare the three event families that could affect shot/rebound logic.
# We include:
# - shot: the main shooting event
# - deflection: likely an offensive redirection event, but we need to confirm
# - block: defensive/pass-or-shot block attempts per the HALO documentation

shot_block_deflection = events[
    events["event_type"].isin(["shot", "deflection", "block"])
].copy()

summary_sbd = (
    shot_block_deflection
    .groupby(["event_type", "outcome", "detail"], dropna=False)
    .agg(
        rows=("sl_event_id", "count"),
        xg_non_null=("sl_xg_all_shots", "count"),
        tracking_rate=("has_tracking_data", "mean"),
        event_player_tracked_rate=("event_player_tracked", "mean"),
    )
    .reset_index()
    .sort_values(["event_type", "rows"], ascending=[True, False])
)

summary_sbd

,event_type,outcome,detail,rows,xg_non_null,tracking_rate,event_player_tracked_rate
4,block,successful,pass,42614,0,0.939292,0.657202
5,block,successful,shot,12276,0,0.940290,0.679945
1,block,failed,pass,5040,0,0.937698,0.678175
3,block,successful,blueline,4368,0,0.943910,0.646749
2,block,failed,shot,1544,0,0.928109,0.669041
0,block,failed,blueline,391,0,0.951407,0.721228
11,deflection,successful,slot,1063,1063,0.930386,0.654751
8,deflection,failed,slot,738,738,0.940379,0.689702
9,deflection,failed,slotblocked,119,119,0.957983,0.663866
10,deflection,successful,outside,88,88,0.931818,0.659091


In [5]:
# Deflection rows deserve their own look because they may inherit xG from a shot,
# represent a redirected shot, or act like a separate chance.

deflections = events[events["event_type"] == "deflection"].copy()

deflections[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "flags",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
        "has_tracking_data",
        "event_player_tracked",
    ]
].head(30)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,event_type,outcome,flags,description,detail,sl_xg_all_shots,x_adj,y_adj,has_tracking_data,event_player_tracked
3703,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1,174.630000000,168,2,"Fogarty, Steven",IA,deflection,successful,"SHOT_FLAGS(1timer, seam, westeast), DEFLECTION...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.293209,81.176186,-1.258823,1,1
4736,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1,1197.600000000,1202,14,"Wagner, Ryan",CHI,deflection,failed,"SHOT_FLAGS(seam, westeast), DEFLECTION_FLAGS(d...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.227750,81.278534,-0.250000,1,0
5475,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2,795.470000000,1942,26,"Lambos, Carson",IA,deflection,failed,"SHOT_FLAGS(westside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.146001,79.761826,1.754799,1,0
5720,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2,996.230000000,2188,32,"Kent Elson, Turner",IA,deflection,failed,"SHOT_FLAGS(eastside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.050937,65.179690,-3.267647,1,1
6415,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,3,480.600000000,2884,42,"Elynuik, Hudson",CHI,deflection,failed,"SHOT_FLAGS(eastside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.255358,82.787370,-0.250000,1,1
7853,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,1,482.030000000,505,7,"Comtois, Maxime",CHI,deflection,successful,"SHOT_FLAGS(seam, eastwest), DEFLECTION_FLAGS(d...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.081976,72.118520,2.261764,1,1
8700,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2,119.570000000,1353,21,"Caamano, Nicholas",TEX,deflection,failed,"SHOT_FLAGS(westside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slotblocked,0.017653,63.065580,15.841175,1,1
9418,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2,819.170000000,2072,28,"Melnick, Josh",CHI,deflection,failed,"SHOT_FLAGS(westside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: OU...,outside,0.043211,79.761826,16.340092,1,1
9504,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2,887.700000000,2159,31,"Comtois, Maxime",CHI,deflection,failed,"SHOT_FLAGS(eastside), DEFLECTION_FLAGS(deflect...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.193376,79.761826,-1.765789,1,1
10107,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,3,228.230000000,2763,46,"McKenzie, Curtis",TEX,deflection,failed,"SHOT_FLAGS(seam, westeast), DEFLECTION_FLAGS(d...",OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,slot,0.542416,85.302060,-3.267647,1,1


## Named Example: Deflection, Recovery, and Goal

This example shows that a deflected shot can generate several related rows:

- the original shot row
- defensive block/pressure context
- the offensive deflection row
- assist rows timestamped to the contributing actions
- goal rows that may occur later in the same sequence

This means we should not treat a single event row as the complete scoring chance. For deflected shots, the deflection row may be the row carrying xG, while the original shot row provides context.

In [6]:
# Locate the exact deflection row first, then use its game/sequence/time.
# This avoids hardcoding the wrong sequence_id.

target_deflection = events.loc[
    (events["sl_event_id"] == 2763)
    & (events["player_name"] == "McKenzie, Curtis")
].iloc[0]

target_game_id = target_deflection["game_id"]
target_sequence_id = target_deflection["sequence_id"]
target_time = target_deflection["period_time"]

cluster_window = events[
    (events["game_id"] == target_game_id)
    & (events["sequence_id"] == target_sequence_id)
    & (events["period_time"].between(target_time - 5, target_time + 10))
].sort_values("period_time")

cluster_window[
    [
        "period",
        "period_time",
        "sl_event_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
    ]
]

,period,period_time,sl_event_id,player_name,team,event_type,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj
10095,3,223.800000000,2750,"Stankoven, Logan",TEX,pass,successful,SOUTH CYCLE +,south,NaN,55.628540,-33.444115
10096,3,224.470000000,2751,"Pouliot, Derrick",TEX,reception,successful,O-ZONE PASS RECEPTION,regular,NaN,27.463829,-14.835293
10097,3,225.270000000,2752,"Pouliot, Derrick",TEX,pass,successful,NORTH CYCLE +,north,NaN,30.481476,-6.788235
10098,3,226.000000000,2753,"Stankoven, Logan",TEX,reception,successful,O-ZONE PASS RECEPTION,regular,NaN,43.055008,-25.397057
10099,3,226.770000000,2754,"Stankoven, Logan",TEX,pass,successful,EAST/WEST PASS+,eastwest,NaN,51.102066,-23.385292
10100,3,226.770000000,2755,"Stankoven, Logan",TEX,assist,successful,2ND ASSIST,second,NaN,51.102066,-23.385292
10101,3,226.830000000,2756,"Melnick, Josh",CHI,block,failed,BLOCK OPPOSITION PASS-,pass,NaN,-55.124893,23.386734
10102,3,227.400000000,2757,"Bourque, Mavrik",TEX,reception,successful,O-ZONE PASS RECEPTION,regular,NaN,68.202070,23.388235
10104,3,227.800000000,2759,"Bourque, Mavrik",TEX,assist,successful,1ST ASSIST,first,NaN,71.219710,20.370588
10103,3,227.800000000,2758,"Bourque, Mavrik",TEX,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,71.219710,20.370588


### Example: Deflection Followed By Loose Puck Recovery And Goal

A wider event window shows that assist rows can appear before the eventual goal and may be timestamped to the contributing pass or shot action.

In this example:

1. Bourque takes a shot that is described as deflected and missed.
2. McKenzie records a failed offensive deflection with non-null xG.
3. McKenzie recovers the loose puck.
4. McKenzie takes a new slot shot.
5. McKenzie scores.

This means assists are useful context, but they should not be used as direct labels for whether a nearby shot became a goal.

In [7]:
# Quantify xG availability by event type and shot/deflection detail.
# This tests the pattern we observed: original deflected shot rows may have null xG,
# while deflection rows carry xG.

xg_availability = (
    events[events["event_type"].isin(["shot", "deflection"])]
    .groupby(["event_type", "detail"], dropna=False)
    .agg(
        rows=("sl_event_id", "count"),
        xg_non_null=("sl_xg_all_shots", "count"),
        xg_missing=("sl_xg_all_shots", lambda s: s.isna().sum()),
        xg_mean=("sl_xg_all_shots", "mean"),
    )
    .reset_index()
)

xg_availability["xg_non_null_rate"] = (
    xg_availability["xg_non_null"] / xg_availability["rows"]
)

xg_availability.sort_values(["event_type", "detail"])

,event_type,detail,rows,xg_non_null,xg_missing,xg_mean,xg_non_null_rate
0,deflection,outside,123,123,0,0.042750,1.000000
1,deflection,outsideblocked,4,4,0,0.043615,1.000000
2,deflection,slot,1801,1801,0,0.148345,1.000000
3,deflection,slotblocked,119,119,0,0.078235,1.000000
4,shot,d2doffboards,1,0,1,NaN,0.000000
5,shot,eastwest,10,0,10,NaN,0.000000
6,shot,north,88,0,88,NaN,0.000000
7,shot,northoffboards,2,0,2,NaN,0.000000
8,shot,outlet,4,0,4,NaN,0.000000
9,shot,outletoffboards,1,0,1,NaN,0.000000


In [8]:
# Check how many shot rows are explicitly described as deflected.
# These may be original shot rows whose xG is assigned to the later deflection row.

deflected_shot_rows = events[
    (events["event_type"] == "shot")
    & (
        events["description"]
        .astype("string")
        .str.contains("DEFLECT", case=False, na=False)
    )
].copy()

deflected_shot_rows.shape

(2047, 24)

In [9]:
deflected_shot_rows[
    [
        "period",
        "period_time",
        "sl_event_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
    ]
].head(20)

,period,period_time,sl_event_id,player_name,team,event_type,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj
3701,1,174.330000000,165,"Toporowski, Luke",IA,shot,successful,DEFLECTED SHOT FOR ONNET,slot,NaN,66.590900,30.426468
4734,1,1196.830000000,1199,"Grimaldi, Rocco",CHI,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,34.505005,31.938236
5473,2,794.900000000,1939,"Bankier, Caedan",IA,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,37.011826,38.469500
5718,2,995.930000000,2185,"Bankier, Caedan",IA,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,51.600280,-29.923530
6413,3,480.170000000,2881,"Sucese, Nate",CHI,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,73.231476,-20.870590
7851,1,481.230000000,502,"Ponomarev, Vasiliy",CHI,shot,successful,DEFLECTED SHOT FOR ONNET,slot,NaN,28.865578,-24.394117
8698,2,119.170000000,1350,"Pouliot, Derrick",TEX,shot,successful,DEFLECTED SHOT FOR BLOCKED,slot,NaN,32.386170,39.479410
9416,2,818.730000000,2069,"Comtois, Maxime",CHI,shot,successful,DEFLECTED SHOT FOR MISSED,north,NaN,49.082413,37.463620
9502,2,887.300000000,2156,"Terry, Chris",CHI,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,65.176530,-14.842262
10103,3,227.800000000,2758,"Bourque, Mavrik",TEX,shot,successful,DEFLECTED SHOT FOR MISSED,slot,NaN,71.219710,20.370588


In [10]:
# Compare xG missingness for explicitly deflected shot rows.

deflected_shot_rows["sl_xg_all_shots"].isna().mean()

np.float64(0.9956033219345384)

In [11]:
# How many shot rows have one of the official shot details from the HALO README?
# This separates normal shot attempts from rows with unexpected detail values.

official_shot_details = ["outside", "outsideblocked", "slot", "slotblocked"]

shots = events[events["event_type"] == "shot"].copy()

shots["is_official_shot_detail"] = shots["detail"].isin(official_shot_details)

shots["is_official_shot_detail"].value_counts(dropna=False)

is_official_shot_detail
True     53834
False      137
Name: count, dtype: int64

In [12]:
# Inspect the unofficial shot-detail rows.
# These are not necessarily useless, but they are unsafe to include without understanding them.

unofficial_shot_detail_rows = shots[~shots["is_official_shot_detail"]].copy()

unofficial_shot_detail_rows[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
    ]
].head(50)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,event_type,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj
9416,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2,818.730000000,2069,28,"Comtois, Maxime",CHI,shot,successful,DEFLECTED SHOT FOR MISSED,north,NaN,49.082413,37.463620
15462,02baa060-7e1d-95ae-3186-749790963b95,1,554.400000000,634,12,"Middleton, Keaton",COL,shot,successful,DEFLECTED SHOT FOR ONNET,stretch,NaN,-52.504520,10.813263
23615,03470158-c116-5442-a415-c1b51c5ad0c3,1,1190.470000000,1357,26,"Priskie, Chase",HER,shot,successful,DEFLECTED SHOT FOR ONNET,eastwest,NaN,-8.244988,17.352942
27202,06795c32-cffd-e9e8-3a6a-5728a7f07a8b,1,1011.200000000,1064,13,"Osmanski, Austin",SPR,shot,successful,DEFLECTED SHOT FOR ONNET,north,NaN,86.307950,26.405882
59607,0c175bcc-4559-6843-a449-5a8d3ba2b6ba,3,813.130000000,3313,48,"Seppala, Peetro",CVF,shot,successful,DEFLECTED SHOT FOR ONNET,north,NaN,-24.438293,22.609917
69202,0e29a214-f00b-3e86-92e1-ce47bd2b42cc,2,455.870000000,1757,25,"Rempal, Sheldon",HEN,shot,successful,DEFLECTED SHOT FOR MISSED,north,NaN,76.249130,-31.935295
69703,0e29a214-f00b-3e86-92e1-ce47bd2b42cc,2,905.500000000,2260,38,"Geertsen, Mason",HEN,shot,successful,DEFLECTED SHOT FOR ONNET,stretch,NaN,-37.423470,26.398914
69745,0e29a214-f00b-3e86-92e1-ce47bd2b42cc,2,951.230000000,2303,38,"Bucheler, Jeremie",SJ,shot,successful,DEFLECTED SHOT FOR ONNET,stretch,NaN,-64.673584,-24.778322
85348,0fe746fa-d504-2de7-f79e-56a6caa336eb,3,293.930000000,2942,46,"Kemp, Phil",BAK,shot,successful,DEFLECTED SHOT FOR ONNET,stretch,NaN,-71.722660,-7.797058
90989,134899f9-5997-ecdc-5464-d183de63323f,1,702.930000000,836,10,"Farrell, Sean",LAV,shot,successful,DEFLECTED SHOT FOR ONNET,north,NaN,71.117360,-27.109997


In [13]:
# Quantify how much of the shot table would be excluded
# if we restrict to official shot details.

excluded_shot_detail_share = len(unofficial_shot_detail_rows) / len(shots)

len(shots), len(unofficial_shot_detail_rows), excluded_shot_detail_share

(53971, 137, 0.002538400251987178)

### Shot Detail Inclusion Rule

The HALO README defines four official `shot.detail` values:

- `outside`
- `outsideblocked`
- `slot`
- `slotblocked`

The event table contains 137 shot rows with other detail values such as pass or transition labels. These represent approximately 0.25% of all shot rows.

For the first analytical build, we will exclude unofficial shot-detail rows from the shot attempt table and document the exclusion. This keeps the shot definition aligned with the official data dictionary and avoids mixing shot attempts with ambiguous event-label artifacts.

### Missing xG On Slot Shot Rows

Most clean shot attempts have complete `sl_xg_all_shots`, but `shot` rows with `detail == "slot"` have 1,901 missing xG values.

Based on event inspection, many of these may be original shots that were later deflected. In those cases, xG appears to be assigned to the `deflection` row rather than the original `shot` row.

We need to test this before deciding whether missing-xG slot shots should be dropped, linked to deflections, or retained as setup context.

In [14]:
# Is missing xG on slot shots mostly explained by deflected-shot descriptions?

slot_shots = events[
    (events["event_type"] == "shot")
    & (events["detail"] == "slot")
].copy()

slot_shots["description_mentions_deflect"] = (
    slot_shots["description"]
    .astype("string")
    .str.contains("DEFLECT", case=False, na=False)
)

missing_xg_slot_shots = slot_shots[slot_shots["sl_xg_all_shots"].isna()].copy()

missing_xg_deflect_summary = (
    missing_xg_slot_shots["description_mentions_deflect"]
    .value_counts(dropna=False)
    .rename_axis("description_mentions_deflect")
    .reset_index(name="rows")
)

missing_xg_deflect_summary

,description_mentions_deflect,rows
0,True,1901


In [15]:
# Inspect missing-xG slot shots that do NOT mention deflection.
# These are the cases that may break our assumption.

missing_xg_slot_non_deflect = missing_xg_slot_shots[
    ~missing_xg_slot_shots["description_mentions_deflect"]
].copy()

missing_xg_slot_non_deflect[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
        "has_tracking_data",
        "event_player_tracked",
    ]
].head(30)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,event_type,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj,has_tracking_data,event_player_tracked


In [16]:
len(missing_xg_slot_non_deflect), len(missing_xg_slot_shots)

(0, 1901)

## Deflection Event Grammar

Before creating an evaluated chance table, we need to understand how HALO encodes deflected shots.

Open questions:

1. Why are all missing-xG original shot rows labeled `detail == "slot"`?
2. Are deflection locations/details reliable, or should we derive outside/slot from coordinates?
3. Do double deflections exist?
4. Does `description` contain both deflection description and original shot description separated by `; ORIGINAL DESCRIPTION:`?

In [17]:
# Start with original shot rows that are missing xG.
# Earlier inspection showed these are slot shots whose descriptions mention deflection.
# Now we inspect whether their coordinates are actually all slot-like.

missing_xg_shots = events[
    (events["event_type"] == "shot")
    & (events["sl_xg_all_shots"].isna())
].copy()

missing_xg_shots[
    ["detail", "description", "x_adj", "y_adj"]
].head(20)

,detail,description,x_adj,y_adj
3701,slot,DEFLECTED SHOT FOR ONNET,66.590900,30.426468
4734,slot,DEFLECTED SHOT FOR MISSED,34.505005,31.938236
5473,slot,DEFLECTED SHOT FOR MISSED,37.011826,38.469500
5718,slot,DEFLECTED SHOT FOR MISSED,51.600280,-29.923530
6413,slot,DEFLECTED SHOT FOR MISSED,73.231476,-20.870590
7851,slot,DEFLECTED SHOT FOR ONNET,28.865578,-24.394117
8698,slot,DEFLECTED SHOT FOR BLOCKED,32.386170,39.479410
9416,north,DEFLECTED SHOT FOR MISSED,49.082413,37.463620
9502,slot,DEFLECTED SHOT FOR MISSED,65.176530,-14.842262
10103,slot,DEFLECTED SHOT FOR MISSED,71.219710,20.370588


In [18]:
# Summarize coordinate ranges for missing-xG shots.
# If these are truly all slot-origin shots, x_adj/y_adj should mostly fall in the slot/home-plate area.
# If many coordinates are outside, then the `detail == slot` label is not reliable for original deflected shots.

missing_xg_shots[["x_adj", "y_adj"]].describe()

,x_adj,y_adj
count,2038.000000,2038.000000
mean,45.702687,-1.245131
std,19.304859,27.087244
min,-95.757460,-40.939330
25%,35.007950,-26.908823
50%,41.439110,-2.926132
75%,57.637806,25.901083
max,97.769226,40.988235


In [19]:
# Compare missing-xG shot coordinates to normal outside and slot shots.

coord_compare = (
    events[
        (events["event_type"] == "shot")
        & (events["detail"].isin(["outside", "slot"]))
    ]
    .assign(
        xg_status=lambda df: df["sl_xg_all_shots"].notna().map({True: "xg_present", False: "xg_missing"})
    )
    .groupby(["detail", "xg_status"])
    .agg(
        rows=("sl_event_id", "count"),
        mean_x=("x_adj", "mean"),
        median_x=("x_adj", "median"),
        mean_abs_y=("y_adj", lambda s: s.abs().mean()),
        median_abs_y=("y_adj", lambda s: s.abs().median()),
        min_x=("x_adj", "min"),
        max_x=("x_adj", "max"),
        min_y=("y_adj", "min"),
        max_y=("y_adj", "max"),
    )
    .reset_index()
)

coord_compare

,detail,xg_status,rows,mean_x,median_x,mean_abs_y,median_abs_y,min_x,max_x,min_y,max_y
0,outside,xg_present,23695,51.555994,48.480286,22.918413,24.394117,-97.364760,97.86774,-42.497055,41.494118
1,slot,xg_missing,1901,46.792316,41.438250,24.292256,26.402939,25.855820,96.25970,-40.939330,40.988235
2,slot,xg_present,15359,70.688574,70.107460,9.885306,8.802940,54.012638,88.82266,-22.985890,22.945206


### Finding: Missing-xG Deflected Shot Origins Are Mislabeled As Slot

Missing-xG original shot rows are overwhelmingly labeled `detail == "slot"`, but their coordinates do not resemble normal slot shots.

Compared with xG-present slot shots, missing-xG slot-labeled shots are much farther from the net and much wider laterally. Their median `x_adj` is around 41 feet and their median absolute `y_adj` is around 26 feet, while normal xG-present slot shots have median `x_adj` around 70 and median absolute `y_adj` around 9.

This suggests that the `detail` label is unreliable for original deflected shot rows. For shot-origin analysis, we should derive outside/slot status geometrically from coordinates rather than trusting `detail`.

In [20]:
# First-pass geometric slot definition.
#
# Rink convention:
# - x_adj points toward the offensive net on the right.
# - Higher x_adj means closer to the offensive goal.
# - y_adj is lateral distance from rink center.
#
# This is a deliberately simple "home plate-ish" proxy:
# - close enough to the net: x_adj >= 54
# - not too wide: abs(y_adj) <= 22
#
# We will refine this later if needed, but it is already more defensible
# than trusting `detail` for deflected original shot rows.

def add_geometric_location_flags(df):
    out = df.copy()
    out["abs_y_adj"] = out["y_adj"].abs()
    out["is_geometric_slot"] = (
        (out["x_adj"] >= 54)
        & (out["abs_y_adj"] <= 22)
    )
    out["geometric_location"] = out["is_geometric_slot"].map(
        {True: "slot", False: "outside"}
    )
    return out

shot_rows_with_geo = add_geometric_location_flags(
    events[events["event_type"] == "shot"]
)

geo_vs_detail = (
    shot_rows_with_geo[
        shot_rows_with_geo["detail"].isin(["outside", "outsideblocked", "slot", "slotblocked"])
    ]
    .assign(
        detail_location=lambda df: df["detail"].replace(
            {
                "outsideblocked": "outside",
                "slotblocked": "slot",
            }
        ),
        xg_status=lambda df: df["sl_xg_all_shots"].notna().map(
            {True: "xg_present", False: "xg_missing"}
        ),
    )
    .groupby(["detail_location", "geometric_location", "xg_status"])
    .agg(rows=("sl_event_id", "count"))
    .reset_index()
)

geo_vs_detail

,detail_location,geometric_location,xg_status,rows
0,outside,outside,xg_present,29943
1,outside,slot,xg_present,3455
2,slot,outside,xg_missing,1713
3,slot,outside,xg_present,626
4,slot,slot,xg_missing,188
5,slot,slot,xg_present,17909


In [21]:
missing_xg_shots_geo = add_geometric_location_flags(missing_xg_shots)

missing_xg_shots_geo["geometric_location"].value_counts()

geometric_location
outside    1842
slot        196
Name: count, dtype: int64

### Decision: Use Coordinate-Derived Shot Location

The official `detail` field is useful, but it should not be the primary source for analytical outside/slot classification.

A simple coordinate-based slot proxy shows that most missing-xG original deflected shots are geometrically outside, despite being labeled `detail == "slot"`. Therefore, for this project:

- `detail` will help identify blocked/unblocked shot categories.
- `x_adj` and `y_adj` will define analytical shot location.
- Deflected original shot rows will preserve their origin coordinates.
- Deflection rows will preserve the redirected chance coordinates and xG.

In [22]:
# Identify deflection rows and look for multiple deflections close together
# in the same game and sequence. This tests whether double deflections exist.

deflections = events[events["event_type"] == "deflection"].copy()

deflections_sorted = deflections.sort_values(
    ["game_id", "sequence_id", "period_time", "sl_event_id"]
).copy()

deflections_sorted["prev_deflection_time"] = (
    deflections_sorted
    .groupby(["game_id", "sequence_id"])["period_time"]
    .shift(1)
)

deflections_sorted["seconds_since_prev_deflection"] = (
    deflections_sorted["period_time"] - deflections_sorted["prev_deflection_time"]
)

nearby_deflection_pairs = deflections_sorted[
    deflections_sorted["seconds_since_prev_deflection"].between(0, 3, inclusive="both")
].copy()

nearby_deflection_pairs.shape

(0, 26)

In [23]:
nearby_deflection_pairs[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
        "seconds_since_prev_deflection",
    ]
].head(30)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj,seconds_since_prev_deflection


In [24]:
# Search deflection descriptions for language that might indicate multiple redirections.
# This is a lightweight check for double-deflection encoding in text.

double_deflection_text = deflections[
    deflections["description"]
    .astype("string")
    .str.contains("DOUBLE|SECOND|2ND|TWICE|REDIRECT", case=False, na=False)
].copy()

double_deflection_text.shape

(0, 24)

In [25]:
double_deflection_text[
    [
        "game_id",
        "period",
        "period_time",
        "sl_event_id",
        "sequence_id",
        "player_name",
        "team",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
    ]
].head(30)

,game_id,period,period_time,sl_event_id,sequence_id,player_name,team,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj


### Double-Deflection Check

Initial checks found no evidence of separately encoded double deflections:

- no deflection rows occurred within 3 seconds of another deflection in the same game and sequence
- no deflection descriptions matched double/second/twice/redirect language

This does not prove double deflections are impossible in hockey; it only means they are not visibly encoded as separate deflection events under these checks.

In [26]:
# Check whether deflection descriptions consistently contain the original shot description
# after a delimiter. This matters because we may be able to recover original shot context
# from the deflection row itself.

deflections["has_original_description"] = (
    deflections["description"]
    .astype("string")
    .str.contains("; ORIGINAL DESCRIPTION:", regex=False, na=False)
)

deflections["has_original_description"].value_counts(dropna=False)

has_original_description
True    2047
Name: count, dtype: Int64

In [27]:
# Split deflection description into the deflection description and original shot description.
# We use n=1 so only the first delimiter is used if the text contains the phrase more than once.

deflection_description_parts = (
    deflections["description"]
    .astype("string")
    .str.split("; ORIGINAL DESCRIPTION:", n=1, expand=True)
)

deflections["deflection_description"] = deflection_description_parts[0].str.strip()
deflections["original_shot_description"] = deflection_description_parts[1].str.strip()

deflections[
    [
        "description",
        "deflection_description",
        "original_shot_description",
    ]
].head(20)

,description,deflection_description,original_shot_description
3703,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR ONNET
4736,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED
5475,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED
5720,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED
6415,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED
7853,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR ONNET
8700,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR BLOCKED
9418,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: OU...,OFFENSIVE DEFLECTION,OUTSIDE SHOT FOR MISSED
9504,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED
10107,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: SL...,OFFENSIVE DEFLECTION,SLOT SHOT FOR MISSED


### Deflection Description Parsing

All 2,047 deflection rows contain the delimiter `; ORIGINAL DESCRIPTION:`.

This means deflection rows include both:

- a deflection event description, usually `OFFENSIVE DEFLECTION`
- the original shot description, such as `SLOT SHOT FOR MISSED` or `OUTSIDE SHOT FOR ONNET`

The original description is useful context, but the actual preceding shot row is still preferable for original shot coordinates.

In [28]:
# Count original shot descriptions embedded in deflection rows.
# This helps us understand whether original deflected shots were described as slot/outside/on-net/missed/blocked.

original_description_counts = (
    deflections["original_shot_description"]
    .value_counts(dropna=False)
    .rename_axis("original_shot_description")
    .reset_index(name="rows")
)

original_description_counts

,original_shot_description,rows
0,SLOT SHOT FOR ONNET,1063
1,SLOT SHOT FOR MISSED,738
2,SLOT SHOT FOR BLOCKED,119
3,OUTSIDE SHOT FOR ONNET,88
4,OUTSIDE SHOT FOR MISSED,35
5,OUTSIDE SHOT FOR BLOCKED,4


In [29]:
# Link each deflection to the nearest preceding shot in the same game and sequence.
#
# Important:
# - sl_event_id is only unique within a game, so game_id must be retained.
# - period_time can behave like an object/decimal, so we convert the delay to float.
# - This is inventory logic, not final production code.

def find_prev_shot_for_deflections(events, deflections):
    shots_only = events[events["event_type"] == "shot"].copy()

    links = []

    for _, dfl in deflections.iterrows():
        candidates = shots_only[
            (shots_only["game_id"] == dfl["game_id"])
            & (shots_only["sequence_id"] == dfl["sequence_id"])
            & (shots_only["period_time"] <= dfl["period_time"])
        ].copy()

        if candidates.empty:
            links.append(
                {
                    "game_id": dfl["game_id"],
                    "sequence_id": dfl["sequence_id"],
                    "deflection_event_id": dfl["sl_event_id"],
                    "prev_shot_event_id": None,
                    "seconds_after_prev_shot": None,
                }
            )
            continue

        prev = candidates.sort_values(["period_time", "sl_event_id"]).iloc[-1]

        links.append(
            {
                "game_id": dfl["game_id"],
                "sequence_id": dfl["sequence_id"],
                "deflection_event_id": dfl["sl_event_id"],
                "prev_shot_event_id": prev["sl_event_id"],
                "seconds_after_prev_shot": float(dfl["period_time"] - prev["period_time"]),
                "prev_shot_description": prev["description"],
                "prev_shot_detail": prev["detail"],
                "prev_shot_x_adj": prev["x_adj"],
                "prev_shot_y_adj": prev["y_adj"],
                "deflection_x_adj": dfl["x_adj"],
                "deflection_y_adj": dfl["y_adj"],
                "deflection_xg": dfl["sl_xg_all_shots"],
                "original_shot_description": dfl["original_shot_description"],
            }
        )

    return pd.DataFrame(links)

deflection_links = find_prev_shot_for_deflections(events, deflections)

deflection_links.head(20)

,game_id,sequence_id,deflection_event_id,prev_shot_event_id,seconds_after_prev_shot,prev_shot_description,prev_shot_detail,prev_shot_x_adj,prev_shot_y_adj,deflection_x_adj,deflection_y_adj,deflection_xg,original_shot_description
0,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2,168,165,0.30,DEFLECTED SHOT FOR ONNET,slot,66.590900,30.426468,81.176186,-1.258823,0.293209,SLOT SHOT FOR ONNET
1,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,14,1202,1199,0.77,DEFLECTED SHOT FOR MISSED,slot,34.505005,31.938236,81.278534,-0.250000,0.227750,SLOT SHOT FOR MISSED
2,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,26,1942,1939,0.57,DEFLECTED SHOT FOR MISSED,slot,37.011826,38.469500,79.761826,1.754799,0.146001,SLOT SHOT FOR MISSED
3,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,32,2188,2185,0.30,DEFLECTED SHOT FOR MISSED,slot,51.600280,-29.923530,65.179690,-3.267647,0.050937,SLOT SHOT FOR MISSED
4,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,42,2884,2881,0.43,DEFLECTED SHOT FOR MISSED,slot,73.231476,-20.870590,82.787370,-0.250000,0.255358,SLOT SHOT FOR MISSED
5,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,7,505,502,0.80,DEFLECTED SHOT FOR ONNET,slot,28.865578,-24.394117,72.118520,2.261764,0.081976,SLOT SHOT FOR ONNET
6,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,21,1353,1350,0.40,DEFLECTED SHOT FOR BLOCKED,slot,32.386170,39.479410,63.065580,15.841175,0.017653,SLOT SHOT FOR BLOCKED
7,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,28,2072,2069,0.44,DEFLECTED SHOT FOR MISSED,north,49.082413,37.463620,79.761826,16.340092,0.043211,OUTSIDE SHOT FOR MISSED
8,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,31,2159,2156,0.40,DEFLECTED SHOT FOR MISSED,slot,65.176530,-14.842262,79.761826,-1.765789,0.193376,SLOT SHOT FOR MISSED
9,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,46,2763,2758,0.43,DEFLECTED SHOT FOR MISSED,slot,71.219710,20.370588,85.302060,-3.267647,0.542416,SLOT SHOT FOR MISSED


### Deflection Linkage Notes

Deflection rows appear to link cleanly to a preceding shot row in the same game and sequence, usually within less than one second.

However, `sl_event_id` is unique within a game, not globally. Any linkage table must include `game_id` alongside `sl_event_id`.

Also, `period_time` may be stored as a decimal/object type, so time differences should be explicitly converted to numeric before summary statistics.

In [30]:
deflection_links["seconds_after_prev_shot"].describe()

count    2047.000000
mean        0.481822
std         0.189651
min         0.100000
25%         0.370000
50%         0.460000
75%         0.560000
max         4.470000
Name: seconds_after_prev_shot, dtype: float64

In [31]:
deflection_links["prev_shot_event_id"].isna().sum()

np.int64(0)

In [32]:
# Inspect the longest shot-to-deflection delays.
# Most deflections happen within about half a second, so long delays deserve review.

deflection_links.sort_values(
    "seconds_after_prev_shot",
    ascending=False
).head(20)

,game_id,sequence_id,deflection_event_id,prev_shot_event_id,seconds_after_prev_shot,prev_shot_description,prev_shot_detail,prev_shot_x_adj,prev_shot_y_adj,deflection_x_adj,deflection_y_adj,deflection_xg,original_shot_description
1635,cc5e0263-9e17-d3a5-7679-5462707f62cc,49,3618,3615,4.47,DEFLECTED SHOT FOR MISSED,north,30.983719,2.217548,38.024887,17.808723,0.002432,OUTSIDE SHOT FOR MISSED
591,4fbdd26f-e9c8-43e6-cbc5-d6b62b42e70d,29,2129,2125,2.30,DEFLECTED SHOT FOR ONNET,ozentry,-8.747932,-34.449997,76.752075,2.767647,0.063406,SLOT SHOT FOR ONNET
255,21c5f143-a77d-b15b-5df5-d3be3f8a1dbc,3,149,146,1.74,DEFLECTED SHOT FOR ONNET,outside,80.775604,-36.964706,86.307950,2.767647,0.691875,SLOT SHOT FOR ONNET
275,246445f4-6269-ac64-b69f-5036da91a2d2,70,3473,3470,1.57,DEFLECTED SHOT FOR ONNET,outsideblocked,36.409700,26.905884,86.200874,8.297058,0.120956,OUTSIDE SHOT FOR ONNET
905,7775334b-46f4-c0ef-31f5-0ab12054a259,68,3801,3798,1.54,DEFLECTED SHOT FOR MISSED,north,46.575592,-5.279411,73.734420,-25.397057,0.027392,OUTSIDE SHOT FOR MISSED
1223,a2466f78-d8ee-a3c7-fe6f-fa9076aad73b,48,3705,3702,1.50,DEFLECTED SHOT FOR MISSED,slot,50.599976,-37.977554,70.717636,-13.836380,0.037498,SLOT SHOT FOR MISSED
426,3cf0528d-a562-f62d-5560-93079528b9c7,52,2592,2590,1.47,DEFLECTED SHOT FOR GOAL,outside,38.421463,8.297058,87.206760,2.764706,0.711449,SLOT SHOT FOR ONNET
748,609644d8-b5b8-8973-339a-53ca2c0d4ce1,41,3267,3262,1.43,DEFLECTED SHOT FOR ONNET,d2doffboards,-95.757460,3.772087,85.698640,11.819092,0.033333,OUTSIDE SHOT FOR ONNET
1622,ca84c203-277e-d969-92f8-cdb12fa44198,17,920,917,1.36,DEFLECTED SHOT FOR MISSED,slot,59.644180,-38.648148,75.235350,2.090088,0.139615,SLOT SHOT FOR MISSED
74,0c175bcc-4559-6843-a449-5a8d3ba2b6ba,48,3316,3313,1.34,DEFLECTED SHOT FOR ONNET,north,-24.438293,22.609917,11.270531,7.298145,0.001511,OUTSIDE SHOT FOR ONNET


In [33]:
# Inspect the full sequence window around the longest deflection delay.

longest_link = deflection_links.sort_values(
    "seconds_after_prev_shot",
    ascending=False
).iloc[0]

long_delay_context = events[
    (events["game_id"] == longest_link["game_id"])
    & (events["sequence_id"] == longest_link["sequence_id"])
    & (
        events["period_time"].between(
            events.loc[
                (events["game_id"] == longest_link["game_id"])
                & (events["sl_event_id"] == longest_link["prev_shot_event_id"]),
                "period_time"
            ].iloc[0] - 1,
            events.loc[
                (events["game_id"] == longest_link["game_id"])
                & (events["sl_event_id"] == longest_link["deflection_event_id"]),
                "period_time"
            ].iloc[0] + 1,
        )
    )
].sort_values("period_time")

long_delay_context[
    [
        "period",
        "period_time",
        "sl_event_id",
        "player_name",
        "team",
        "event_type",
        "outcome",
        "description",
        "detail",
        "sl_xg_all_shots",
        "x_adj",
        "y_adj",
    ]
]

,period,period_time,sl_event_id,player_name,team,event_type,outcome,description,detail,sl_xg_all_shots,x_adj,y_adj
1405235,3,1149.430000000,3615,"Evans, Ryker",CVF,shot,successful,DEFLECTED SHOT FOR MISSED,north,NaN,30.983719,2.217548
1405236,3,1153.870000000,3617,"Anderson-Dolan, Jaret",ONT,pressure,undetermined,SHOT PRESSURE,shot,NaN,-47.581482,-14.335295
1405237,3,1153.900000000,3618,"Poturalski, Andrew",CVF,deflection,failed,OFFENSIVE DEFLECTION; ORIGINAL DESCRIPTION: OU...,outside,0.002432,38.024887,17.808723


In [34]:
# Count deflections by shot-to-deflection delay bucket.
# This helps decide whether we need a maximum-linkage threshold.

deflection_links["delay_bucket"] = pd.cut(
    deflection_links["seconds_after_prev_shot"],
    bins=[0, 0.5, 1, 2, 3, 5],
    labels=["<=0.5s", "0.5-1s", "1-2s", "2-3s", "3-5s"],
    include_lowest=True,
)

deflection_links["delay_bucket"].value_counts().sort_index()

delay_bucket
<=0.5s    1362
0.5-1s     665
1-2s        18
2-3s         1
3-5s         1
Name: count, dtype: int64

### Deflection Linkage Timing Rule

Most deflections occur very shortly after the preceding shot row. In this dataset, 2,027 of 2,047 deflections occur within 1 second of the prior shot in the same game and sequence, and 2,045 of 2,047 occur within 2 seconds.

The two cases above 2 seconds appear unusual and may reflect event timing artifacts or ambiguous linkage.

For the first build, we will link deflections to the nearest prior shot in the same game and sequence when the delay is no more than 2 seconds. Longer-delay cases will be flagged for review rather than treated as standard deflection links.

In [35]:
# Apply a first-pass deflection linkage quality flag.
# We use 2 seconds as a conservative threshold based on the empirical delay distribution.

DEFLECTION_LINK_MAX_SECONDS = 2.0

deflection_links["valid_deflection_link"] = (
    deflection_links["seconds_after_prev_shot"] <= DEFLECTION_LINK_MAX_SECONDS
)

deflection_links["valid_deflection_link"].value_counts()

valid_deflection_link
True     2045
False       2
Name: count, dtype: int64

In [36]:
# Review invalid links separately.
# These should not drive the main analysis.

invalid_deflection_links = deflection_links[
    ~deflection_links["valid_deflection_link"]
].copy()

invalid_deflection_links

,game_id,sequence_id,deflection_event_id,prev_shot_event_id,seconds_after_prev_shot,prev_shot_description,prev_shot_detail,prev_shot_x_adj,prev_shot_y_adj,deflection_x_adj,deflection_y_adj,deflection_xg,original_shot_description,delay_bucket,valid_deflection_link
591,4fbdd26f-e9c8-43e6-cbc5-d6b62b42e70d,29,2129,2125,2.30,DEFLECTED SHOT FOR ONNET,ozentry,-8.747932,-34.449997,76.752075,2.767647,0.063406,SLOT SHOT FOR ONNET,2-3s,False
1635,cc5e0263-9e17-d3a5-7679-5462707f62cc,49,3618,3615,4.47,DEFLECTED SHOT FOR MISSED,north,30.983719,2.217548,38.024887,17.808723,0.002432,OUTSIDE SHOT FOR MISSED,3-5s,False


### Parking Lot: Refined Shot-Origin Geometry

The first-pass geometric slot flag is only a diagnostic proxy. Later, we may test whether outside-shot value differs by shot-origin region, such as:

- blue-line shots
- flank / wall shots
- high slot shots
- low-angle shots
- point shots through traffic

This can be handled with an offensive-zone grid or a home-plate polygon. For now, the immediate goal is to establish a defensible first analytical dataset.